# MemoryVerse Empirical Extension Study (v6)
**Manuscript:** *When Do Text Proxies Suffice? A Regime Analysis of Sparse, Dense, and Native-Visual Retrieval Under Proxy Noise*

### Kaggle Execution Instructions:
1. **Accelerator:** Select **GPU T4 x2** or **P100** (Session > Accelerator > GPU).
2. **Internet:** Turn **Internet ON** (Session options > Internet ON) for package installs and HuggingFace models.
3. **Secrets (Optional):** If running gated models, add `HF_TOKEN` in Add-ons > Secrets.
4. **Execution:** Click **Save Version** -> Choose **Save & Run All (Commit)**.
5. **Real-Time Heartbeat:** A heartbeat prints every 60 seconds along with live GPU progress updates.
6. **Output Retrieval:** When the commit completes, navigate to the **Output** tab to download `results_v6_final.zip`.

In [ ]:
# 1. Environment Diagnostics & Package Installation
import os
import sys
import subprocess
from pathlib import Path

print("=== Python & Platform Diagnostics ===", flush=True)
print(f"Python Version: {sys.version}", flush=True)
print(f"Working Directory: {os.getcwd()}", flush=True)

# Direct caches to /tmp so model weights never count toward /kaggle/working
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["TORCH_HOME"] = "/tmp/hf_cache"
os.environ["PYTHONUNBUFFERED"] = "1"
Path("/tmp/hf_cache").mkdir(parents=True, exist_ok=True)

# Force-upgrade torchao and install required dependencies
print("Installing / updating dependencies with torchao compatibility...", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.16.0", "colpali-engine>=0.3.0", "accelerate", "bitsandbytes", "sentence-transformers", "timm", "einops", "datasets>=2.14.0"])


In [ ]:
# 2. GPU Hardware Verification
import torch
print(f"PyTorch Version: {torch.__version__}", flush=True)
print(f"CUDA Available: {torch.cuda.is_available()}", flush=True)
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}", flush=True)
    print(f"Device Count: {torch.cuda.device_count()}", flush=True)
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB", flush=True)
    subprocess.run(["nvidia-smi"])
else:
    print("Warning: No GPU detected. Running in CPU fallback mode.", flush=True)


In [ ]:
# 3. Deploy Runner Script & Execute with Live Heartbeat and Streaming Output
import time
import threading
import base64
from pathlib import Path

# Locate dataset root
kaggle_input = Path("/kaggle/input")
dataset_root = None
if kaggle_input.exists():
    for p in kaggle_input.glob("**/public_benchmark_v2.json"):
        curr = p.parent
        while curr != curr.parent:
            if (curr / "ai_engine" / "rag" / "data" / "v2" / "public_benchmark_v2.json").exists():
                dataset_root = curr
                break
            curr = curr.parent
        if dataset_root:
            break

if not dataset_root:
    dataset_root = Path(".").resolve()

print(f"Detected dataset root: {dataset_root}", flush=True)

# Deploy latest optimized runner to /kaggle/working/kaggle_run_v6.py
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
working_dir.mkdir(parents=True, exist_ok=True)
runner_path = working_dir / "kaggle_run_v6.py"

runner_b64 = "..."
with open(runner_path, "wb") as f:
    f.write(base64.b64decode(runner_b64))

print(f"Deployed updated master runner ({runner_path.stat().st_size:,} bytes) to {runner_path}", flush=True)

# ─────────────────────────────────────────────────────────────────────────────
# ACTIVE 1-MINUTE HEARTBEAT THREAD
# ─────────────────────────────────────────────────────────────────────────────
stop_heartbeat = threading.Event()
start_time = time.time()

def heartbeat_loop():
    hb_count = 0
    while not stop_heartbeat.wait(60.0):
        hb_count += 1
        elapsed_min = (time.time() - start_time) / 60.0
        utc_str = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
        print(f"\n❤️  [HEARTBEAT #{hb_count}] Pipeline active for {elapsed_min:.1f} min ({utc_str}) | GPU active", flush=True)

hb_thread = threading.Thread(target=heartbeat_loop, daemon=True)
hb_thread.start()

# Execute with live unbuffered streaming (-u flag + bufsize=1)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
cmd = [sys.executable, "-u", str(runner_path), "--run", "all", "--data_dir", str(dataset_root)]
print(f"Launching runner command: {" ".join(cmd)}", flush=True)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
stop_heartbeat.set()
total_elapsed = (time.time() - start_time) / 60.0
print(f"\n[EXECUTION FINISHED] Return code: {proc.returncode} | Total elapsed: {total_elapsed:.2f} min", flush=True)
if proc.returncode != 0:
    print("Warning: runner exited with non-zero exit code.", flush=True)


In [ ]:
# 4. Output Archive Verification
import hashlib
working = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
print("=== Output Directory Contents ===")
for f in sorted(working.glob("*.zip")):
    size_mb = f.stat().st_size / (1024 * 1024)
    with open(f, "rb") as fp:
        sha = hashlib.sha256(fp.read()).hexdigest()
    print(f"  {f.name:30} : {size_mb:.2f} MB | SHA-256: {sha}")
print("\nExecution complete. Download results_v6_final.zip from the Kaggle Output tab.")